# 03 · Отчёт
Короткий отчёт по накопленным данным: то, что показывается на собеседовании. Проект — мониторинг и исследование, не торговая стратегия.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
from storage.db import connect

def q(sql, params=()):
    """Запрос к базе -> DataFrame. Соединение только на чтение и сразу закрывается."""
    with connect(read_only=True) as con:
        return con.execute(sql, list(params)).df()

## Сколько раз каждая биржа была самой дорогой

In [ ]:
q("""SELECT symbol, exchange_hi, count(*) AS slots FROM spreads GROUP BY ALL""").pivot(index="symbol", columns="exchange_hi", values="slots").fillna(0).astype(int)

## Какая монета самая волатильная по funding

In [ ]:
q("""SELECT symbol, round(avg(funding_rate) * 100, 4) AS avg_pct_8h, round(stddev_samp(funding_rate) * 100, 4) AS std_pct_8h,
            round(avg(funding_rate) * 3 * 365 * 100, 1) AS avg_annualized_pct
     FROM funding GROUP BY symbol ORDER BY std_pct_8h DESC""")

## OI и цена: что происходило с позиционированием
OI растёт и цена растёт — новые лонги; OI растёт и цена падает — новые шорты.

In [ ]:
q("""
WITH oi AS (SELECT symbol, ts, sum(oi_usd) AS oi FROM open_interest GROUP BY ALL),
     px AS (SELECT symbol, ts, avg(close) AS close FROM klines GROUP BY ALL),
     j AS (SELECT oi.symbol, oi.ts, oi.oi, px.close,
                  oi.oi / lag(oi.oi, 24) OVER w - 1 AS oi_chg, px.close / lag(px.close, 24) OVER w - 1 AS px_chg
           FROM oi JOIN px USING (symbol, ts) WINDOW w AS (PARTITION BY oi.symbol ORDER BY oi.ts))
SELECT symbol, round(oi_chg * 100, 1) AS oi_24h_pct, round(px_chg * 100, 1) AS price_24h_pct,
       CASE WHEN oi_chg > 0 AND px_chg > 0 THEN 'новые лонги' WHEN oi_chg > 0 AND px_chg < 0 THEN 'новые шорты'
            WHEN oi_chg < 0 AND px_chg > 0 THEN 'закрытие шортов' ELSE 'закрытие лонгов' END AS reading
FROM j QUALIFY row_number() OVER (PARTITION BY symbol ORDER BY ts DESC) = 1 ORDER BY symbol""")

## Корреляции

In [ ]:
from analysis import correlations
with connect(read_only=True) as con:
    corr = correlations.correlation_matrix(con, 7)
corr.round(2).style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1)

## Алерты: сколько, какие, как быстро закрывались

In [ ]:
q("""SELECT type, count(*) AS episodes, count(resolved_ts) AS closed,
            round(median(epoch(resolved_ts - ts)) / 60) AS median_minutes
     FROM anomalies GROUP BY type ORDER BY episodes DESC""")

## Правило против модели (режим рынка)
Заполняется после каждой утренней сводки. Вопрос эксперимента: где правило лучше, где модель?

In [ ]:
log = q("SELECT * FROM regime_log ORDER BY ts DESC")
print("совпадений:", (log.rule_regime == log.llm_regime).sum(), "из", log.llm_regime.notna().sum())
log.head(14)

## Сколько стоил AI-слой

In [ ]:
q("""SELECT ts::DATE AS day, kind, count(*) AS calls, sum(in_tokens) AS tokens_in, sum(out_tokens) AS tokens_out,
            round(sum(cost_usd), 4) AS cost_usd FROM ai_calls GROUP BY ALL ORDER BY day DESC, kind""")